# Week 18, Day 5: Many tests at once: false discoveries and double dipping

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w018/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Implement Benjamini-Hochberg

Write `benjamini_hochberg(p, q)` returning a boolean array of discoveries. Then simulate 300 studies of 1,000 tests, 10% of them real (z shifted by 3), and record for BH and for Bonferroni the false discovery proportion and the fraction of real effects found.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt
rng = np.random.default_rng(1995)
Phi = np.vectorize(lambda z: 0.5 * (1 + erf(z / sqrt(2))))

def benjamini_hochberg(p, q):
    m = len(p)
    order = np.argsort(p)
    passed = p[order] <= np.arange(1, m + 1) / m * q
    k = ...                                       # largest rank that passes (0 if none)
    out = np.zeros(m, bool)
    out[order[:k]] = True
    return out

fdp_bh, power_bh, power_bon = [], [], []
for _ in range(300):
    real = np.arange(1000) < 100
    z = rng.normal(size=1000) + 3 * real
    p = 2 * (1 - Phi(np.abs(z)))
    d_bh = benjamini_hochberg(p, 0.05)
    d_bon = ...
    fdp_bh.append(np.sum(d_bh & ~real) / max(d_bh.sum(), 1))
    power_bh.append(np.mean(d_bh[real])); power_bon.append(np.mean(d_bon[real]))
print(f"BH: false discovery proportion {np.mean(fdp_bh):.3f}, real effects found {np.mean(power_bh):.2f};  Bonferroni finds {np.mean(power_bon):.2f}")

In [ ]:
import numpy as np
def test_bh_example():
    "benjamini_hochberg finds the right discoveries in a small example"
    p = np.array([0.001, 0.008, 0.039, 0.041, 0.042, 0.06, 0.074, 0.205])
    assert set(np.flatnonzero(benjamini_hochberg(p, 0.05))) == {0, 1}

def test_fdr():
    "over 300 simulated studies, BH keeps the false discovery proportion near or below 0.05 (below 0.07)"
    assert np.mean(fdp_bh) < 0.07

def test_power():
    "BH finds more real effects than Bonferroni"
    assert np.mean(power_bh) > np.mean(power_bon) + 0.1

test_bh_example()
test_fdr()
test_power()
print("All checks passed")

### Exercise: A response from pure noise

200 neurons have no true response; each has 20 trials of response-minus-baseline differences with sd 1. Select the 20 neurons with the largest mean difference over all trials and report their average over those same trials (`circular`). Then select on trials 1 to 10 and report on trials 11 to 20 (`split`).

In [ ]:
r6 = np.random.default_rng(2009)
D = r6.normal(size=(20, 200))                 # trials x neurons, pure noise
top = np.argsort(D.mean(axis=0))[-20:]
circular = ...
top_half = np.argsort(D[:10].mean(axis=0))[-20:]
split = ...
print(f"circular estimate {circular:.3f};  split-half estimate {split:.3f}")

In [ ]:
import numpy as np
def test_circular():
    "the circular estimate is clearly positive although no neuron responds (above 0.3)"
    assert circular > 0.3

def test_split():
    "the split-half estimate is close to zero (within 0.1)"
    assert abs(split) < 0.1

test_circular()
test_split()
print("All checks passed")

### Exercise: Five reasonable windows

Simulate 4,000 null experiments, each with 30 trials of a neuron's spike counts in 5 overlapping time windows (counts in adjacent windows share spikes, so they are correlated). For each experiment, test the first window only (`fp_single`), and also report the smallest p-value across the five windows (`fp_best`), counting false positives at 0.05.

In [ ]:
fp1, fp5 = [], []
for _ in range(4000):
    bins = r6.normal(size=(30, 7))                 # 7 small time bins of noise per trial
    windows = np.column_stack([bins[:, i:i + 3].sum(axis=1) for i in range(5)])    # 5 overlapping windows
    z = windows.mean(axis=0) / (windows.std(axis=0, ddof=1) / np.sqrt(30))
    p = 2 * (1 - Phi(np.abs(z)))
    fp1.append(...)
    fp5.append(...)
fp_single, fp_best = np.mean(fp1), np.mean(fp5)
print(f"false positive rate: one pre-chosen window {fp_single:.3f}, best of five {fp_best:.3f}")

In [ ]:
import numpy as np
def test_single():
    "a single pre-chosen window gives about 5% false positives (within 2.5 points)"
    assert abs(fp_single - 0.05) < 0.025

def test_best_of_five():
    "reporting the best of five windows gives at least twice as many"
    assert fp_best > 2 * fp_single

test_single()
test_best_of_five()
print("All checks passed")